# Chapter 15: Introduction to Agentic AI — Building the First GAS Agent

> **Part 5: Agentic AI Systems** | Guardian Angel System (GAS) — Multi-Agent Diabetic Patient Supervision

---

## 🎯 Chapter Goal

By the end of this chapter you will:

1. **Understand** the perceive-reason-act loop that underlies every agentic AI system
2. **Build** a minimal ReAct agent using OpenAI function calling
3. **Define** the `GASState` TypedDict that will grow across all chapters in Part 5
4. **Introduce** the full Guardian Angel System (GAS) architecture — 5 agents, their roles, and how they connect

### The Guardian Angel System (GAS)

Throughout Part 5, we build a **multi-agent diabetic patient supervision system** called the Guardian Angel System. It continuously monitors a patient's continuous glucose monitor (CGM) data and coordinates 5 specialized agents:

| Agent | Role |
|-------|------|
| **Supervisor** | LangGraph orchestrator — routes between agents |
| **Monitor** | Reads CGM sensor data, detects anomalies |
| **Advisor** | RAG + memory-grounded clinical recommendations |
| **Safety Critic** | Reward-model-based output validation |
| **Caregiver Notifier** | Escalation to humans via A2A protocol |

This chapter builds the **skeleton** of the Monitor Agent.

## 📖 Theory Recap

### The Perceive-Reason-Act Loop

Every agentic AI system is built on a fundamental loop:

```
┌─────────────────────────────────────────────────────┐
│                                                     │
│   PERCEIVE ──► REASON ──► ACT ──► (back to PERCEIVE)│
│                                                     │
└─────────────────────────────────────────────────────┘
```

- **Perceive**: Read sensor data, tool outputs, user messages, memory
- **Reason**: Use an LLM to decide what to do next (chain-of-thought, ReAct)
- **Act**: Call a tool, update state, send a message, or terminate

### ReAct: Reasoning + Acting

ReAct (Yao et al., 2022) interleaves **Thought** and **Action** steps:

```
Thought: The glucose is 45 mg/dL — this is severe hypoglycemia.
Action: read_glucose()
Observation: {"value": 45, "trend": "falling"}
Thought: Falling trend makes this critical. I must alert immediately.
Action: send_alert(level="critical", message="Severe hypoglycemia")
Observation: Alert sent to caregiver.
Thought: Alert sent. Recommend 15g fast carbs per Rule of 15.
Final Answer: Critical hypoglycemia detected. Alert sent. Recommend 15g fast-acting carbs.
```

### Tool Use and Function Calling

Modern LLMs support **structured tool calls** — the model outputs a JSON object specifying:
- Which tool to call
- What arguments to pass

The runtime executes the tool and returns the result as an observation. This is the foundation of all agentic systems.

### Agent Taxonomy

| Type | Description | GAS Example |
|------|-------------|-------------|
| **Autonomous** | Acts independently, no human approval | Monitor Agent (continuous loop) |
| **Transactional** | Executes discrete tasks with clear inputs/outputs | Advisor Agent (query → recommendation) |
| **Collaborative** | Works with other agents, shares state | All 5 GAS agents via GASState |
| **Supervisory** | Orchestrates other agents | Supervisor Agent (LangGraph) |

### GAS Architecture Overview

```
                    ┌─────────────────────┐
                    │   SUPERVISOR AGENT  │
                    │  (LangGraph Router) │
                    └──────────┬──────────┘
                               │ GASState
              ┌────────────────┼────────────────┐
              ▼                ▼                ▼
    ┌──────────────┐  ┌──────────────┐  ┌──────────────┐
    │   MONITOR    │  │   ADVISOR    │  │    SAFETY    │
    │    AGENT     │  │    AGENT     │  │    CRITIC    │
    │  CGM + Alerts│  │  RAG + Memory│  │  Validation  │
    └──────────────┘  └──────────────┘  └──────────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │  CAREGIVER NOTIFIER │
                    │   (A2A Escalation)  │
                    └─────────────────────┘
```

**Data flow**: CGM sensor → Monitor → Supervisor → Advisor → Safety Critic → (if approved) Caregiver Notifier

## 🔨 Build It

### Step 1: Install Dependencies

In [ ]:
# Install required packages
# %pip install openai python-dotenv

### Step 2: Environment Setup

In [ ]:
import os
import json
import datetime
from typing import TypedDict, Optional, List, Any
from dataclasses import dataclass, field

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL

### Step 3: Define the GASState Schema

This TypedDict is the **shared state** that flows through all 5 GAS agents. We define it once here and reuse it across all Part 5 chapters.

In [ ]:
# ─── Core Data Classes ───────────────────────────────────────────────────────

@dataclass
class PatientProfile:
    """Static patient information."""
    patient_id: str
    name: str
    age: int
    diabetes_type: str          # "T1D" or "T2D"
    target_glucose_low: float   # mg/dL — lower bound of target range
    target_glucose_high: float  # mg/dL — upper bound of target range
    insulin_sensitivity: float  # mg/dL drop per 1 unit of insulin
    carb_ratio: float           # grams of carbs per 1 unit of insulin
    emergency_contact: str = "caregiver@example.com"


@dataclass
class GlucoseReading:
    """A single CGM reading."""
    value: float                # mg/dL
    timestamp: datetime.datetime
    trend: str                  # "rising", "falling", "stable", "rapidly_rising", "rapidly_falling"
    sensor_id: str
    confidence: float = 1.0     # 0.0–1.0 sensor confidence


@dataclass
class GASMemory:
    """Multi-type memory store for the GAS system."""
    recent_readings: List[GlucoseReading] = field(default_factory=list)  # Working memory
    past_episodes: List[dict] = field(default_factory=list)              # Episodic memory
    patient_notes: List[str] = field(default_factory=list)               # Semantic memory


# ─── GASState TypedDict ───────────────────────────────────────────────────────

class GASState(TypedDict):
    """Shared state flowing through all 5 GAS agents."""
    patient: Any                 # PatientProfile
    current_reading: Any         # GlucoseReading
    memory: Any                  # GASMemory
    alert_level: str             # "none" | "low" | "medium" | "high" | "critical"
    recommendation: str          # Latest recommendation text
    safety_approved: bool        # Has Safety Critic approved the recommendation?
    caregiver_notified: bool     # Has caregiver been notified?
    agent_messages: List[dict]   # Trace of all agent messages
    next_agent: str              # Which agent runs next


print("✅ GASState schema defined")
print("\nGASState fields:")
for field_name, field_type in GASState.__annotations__.items():
    print(f"  {field_name}: {field_type}")

### Step 4: Create the Synthetic Patient — Alice

In [ ]:
# Create Alice — our synthetic T1D patient used throughout Part 5
alice = PatientProfile(
    patient_id="patient-001",
    name="Alice",
    age=34,
    diabetes_type="T1D",
    target_glucose_low=70.0,    # ADA target: 70–180 mg/dL
    target_glucose_high=180.0,
    insulin_sensitivity=50.0,   # 1 unit drops glucose by 50 mg/dL
    carb_ratio=15.0,            # 1 unit covers 15g carbs
    emergency_contact="bob.smith@example.com"
)

print("👤 Patient Profile: Alice")
print(f"   Age: {alice.age}, Type: {alice.diabetes_type}")
print(f"   Target range: {alice.target_glucose_low}–{alice.target_glucose_high} mg/dL")
print(f"   Insulin sensitivity: {alice.insulin_sensitivity} mg/dL per unit")
print(f"   Carb ratio: {alice.carb_ratio}g carbs per unit")
print(f"   Emergency contact: {alice.emergency_contact}")

# Create a critical hypoglycemia reading
critical_reading = GlucoseReading(
    value=45.0,
    timestamp=datetime.datetime.now(),
    trend="falling",
    sensor_id="dexcom-g7-001",
    confidence=0.98
)

print(f"\n🩸 Current Reading: {critical_reading.value} mg/dL ({critical_reading.trend})")
print(f"   Sensor: {critical_reading.sensor_id}, Confidence: {critical_reading.confidence}")
print(f"   Status: {'⚠️ CRITICAL HYPOGLYCEMIA' if critical_reading.value < 54 else 'Normal'}")

### Step 5: Define the GAS Tools

In [ ]:
# ─── Tool Definitions (OpenAI function calling format) ────────────────────────

GAS_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_glucose",
            "description": "Read the current CGM glucose value and trend for the patient.",
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_id": {
                        "type": "string",
                        "description": "The patient ID to read glucose for"
                    }
                },
                "required": ["patient_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_guidelines",
            "description": "Look up ADA clinical guidelines for a given glucose level. Returns recommended actions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "glucose_value": {
                        "type": "number",
                        "description": "Glucose value in mg/dL"
                    },
                    "trend": {
                        "type": "string",
                        "description": "Glucose trend: rising, falling, or stable"
                    }
                },
                "required": ["glucose_value"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_alert",
            "description": "Send an alert to the patient's caregiver with a specified urgency level.",
            "parameters": {
                "type": "object",
                "properties": {
                    "level": {
                        "type": "string",
                        "enum": ["low", "medium", "high", "critical"],
                        "description": "Alert urgency level"
                    },
                    "message": {
                        "type": "string",
                        "description": "Alert message to send to caregiver"
                    },
                    "patient_id": {
                        "type": "string",
                        "description": "Patient ID"
                    }
                },
                "required": ["level", "message", "patient_id"]
            }
        }
    }
]

print(f"✅ Defined {len(GAS_TOOLS)} GAS tools:")
for tool in GAS_TOOLS:
    fn = tool["function"]
    print(f"  🔧 {fn['name']}: {fn['description'][:60]}...")

### Step 6: Implement Mock Tool Execution

In [ ]:
# ─── Mock Tool Implementations ────────────────────────────────────────────────

# Simulated CGM data store
_MOCK_CGM_DATA = {
    "patient-001": {"value": 45.0, "trend": "falling", "sensor_id": "dexcom-g7-001"}
}

_ALERT_LOG = []  # Track sent alerts


def tool_read_glucose(patient_id: str) -> dict:
    """Mock: Read CGM glucose value."""
    data = _MOCK_CGM_DATA.get(patient_id, {"value": 100.0, "trend": "stable"})
    return {
        "patient_id": patient_id,
        "glucose_value": data["value"],
        "trend": data["trend"],
        "unit": "mg/dL",
        "timestamp": datetime.datetime.now().isoformat(),
        "sensor_id": data.get("sensor_id", "unknown")
    }


def tool_check_guidelines(glucose_value: float, trend: str = "stable") -> dict:
    """Mock: Look up ADA guidelines for a glucose level."""
    if glucose_value < 54:
        severity = "CRITICAL"
        alert_level = "critical"
        action = (
            "SEVERE HYPOGLYCEMIA (< 54 mg/dL). "
            "If conscious: administer 15-20g fast-acting carbohydrates immediately. "
            "If unconscious: administer glucagon or call 911. "
            "Recheck glucose in 15 minutes (Rule of 15). "
            "Do NOT give insulin."
        )
    elif glucose_value < 70:
        severity = "LOW"
        alert_level = "high"
        action = (
            "HYPOGLYCEMIA (54–70 mg/dL). "
            "Consume 15g fast-acting carbs (4 glucose tablets, 4oz juice). "
            "Recheck in 15 minutes. Repeat if still below 70."
        )
    elif glucose_value <= 180:
        severity = "NORMAL"
        alert_level = "none"
        action = "Glucose within target range (70–180 mg/dL). Continue current management."
    elif glucose_value <= 250:
        severity = "HIGH"
        alert_level = "medium"
        action = (
            "HYPERGLYCEMIA (180–250 mg/dL). "
            "Check for missed insulin dose. Consider correction dose per ISF. "
            "Increase water intake. Recheck in 2 hours."
        )
    else:
        severity = "CRITICAL HIGH"
        alert_level = "critical"
        action = (
            "SEVERE HYPERGLYCEMIA (> 250 mg/dL). "
            "Check ketones immediately. If ketones > 0.6 mmol/L, contact physician. "
            "Administer correction insulin per protocol. Hydrate aggressively."
        )

    # Adjust for trend
    trend_note = ""
    if trend == "falling" and glucose_value < 100:
        trend_note = " FALLING TREND: Act immediately — glucose dropping rapidly."
    elif trend == "rising" and glucose_value > 150:
        trend_note = " RISING TREND: Monitor closely — glucose increasing."

    return {
        "glucose_value": glucose_value,
        "severity": severity,
        "alert_level": alert_level,
        "recommended_action": action + trend_note,
        "source": "ADA Standards of Medical Care in Diabetes 2024"
    }


def tool_send_alert(level: str, message: str, patient_id: str) -> dict:
    """Mock: Send alert to caregiver."""
    alert = {
        "status": "sent",
        "alert_id": f"alert-{len(_ALERT_LOG)+1:04d}",
        "level": level,
        "message": message,
        "patient_id": patient_id,
        "recipient": alice.emergency_contact,
        "timestamp": datetime.datetime.now().isoformat(),
        "delivery_method": "SMS + push notification"
    }
    _ALERT_LOG.append(alert)
    return alert


# Tool dispatcher
TOOL_REGISTRY = {
    "read_glucose": tool_read_glucose,
    "check_guidelines": tool_check_guidelines,
    "send_alert": tool_send_alert,
}


def execute_tool(tool_name: str, tool_args: dict) -> str:
    """Execute a tool by name and return JSON string result."""
    if tool_name not in TOOL_REGISTRY:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})
    try:
        result = TOOL_REGISTRY[tool_name](**tool_args)
        return json.dumps(result, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})


print("✅ Mock tool implementations ready")
print("\nTest: read_glucose for patient-001")
print(execute_tool("read_glucose", {"patient_id": "patient-001"}))

### Step 7: Build the ReAct Agent

In [ ]:
# ─── ReAct Agent Implementation ───────────────────────────────────────────────

SYSTEM_PROMPT = """You are the GAS Monitor Agent — a medical AI assistant that monitors 
diabetic patients' continuous glucose monitor (CGM) data.

Your job:
1. Read the patient's current glucose level
2. Check clinical guidelines for that glucose level
3. Send an alert if the situation is dangerous
4. Provide a clear recommendation

Always use tools to gather information before making recommendations.
Be precise, clinical, and safety-first."""


def run_react_agent_mock(patient_id: str, scenario: str) -> dict:
    """
    Mock ReAct agent that simulates the full reasoning trace
    without requiring an OpenAI API key.
    """
    print(f"\n{'='*60}")
    print(f"🤖 GAS Monitor Agent — Mock ReAct Trace")
    print(f"   Scenario: {scenario}")
    print(f"{'='*60}")

    messages = []
    trace = []

    # ── Step 1: Thought + Tool Call: read_glucose ──
    thought_1 = f"I need to check the current glucose level for patient {patient_id}."
    print(f"\n💭 Thought: {thought_1}")
    print(f"🔧 Action: read_glucose(patient_id='{patient_id}')")

    obs_1 = execute_tool("read_glucose", {"patient_id": patient_id})
    obs_1_data = json.loads(obs_1)
    print(f"👁️  Observation: glucose={obs_1_data['glucose_value']} mg/dL, trend={obs_1_data['trend']}")

    trace.append({"thought": thought_1, "action": "read_glucose", "observation": obs_1_data})

    # ── Step 2: Thought + Tool Call: check_guidelines ──
    glucose = obs_1_data["glucose_value"]
    trend = obs_1_data["trend"]
    thought_2 = (
        f"Glucose is {glucose} mg/dL with a {trend} trend. "
        f"{'This is critically low — below 54 mg/dL threshold.' if glucose < 54 else 'I need to check guidelines.'} "
        f"Let me look up the ADA guidelines."
    )
    print(f"\n💭 Thought: {thought_2}")
    print(f"🔧 Action: check_guidelines(glucose_value={glucose}, trend='{trend}')")

    obs_2 = execute_tool("check_guidelines", {"glucose_value": glucose, "trend": trend})
    obs_2_data = json.loads(obs_2)
    print(f"👁️  Observation: severity={obs_2_data['severity']}, alert_level={obs_2_data['alert_level']}")
    print(f"   Action: {obs_2_data['recommended_action'][:80]}...")

    trace.append({"thought": thought_2, "action": "check_guidelines", "observation": obs_2_data})

    # ── Step 3: Thought + Tool Call: send_alert (if needed) ──
    alert_result = None
    if obs_2_data["alert_level"] in ["high", "critical"]:
        thought_3 = (
            f"This is a {obs_2_data['alert_level'].upper()} situation. "
            f"I must immediately alert the caregiver. "
            f"The falling trend makes this even more urgent."
        )
        alert_msg = (
            f"URGENT: Patient {patient_id} glucose is {glucose} mg/dL ({trend}). "
            f"{obs_2_data['recommended_action'][:100]}"
        )
        print(f"\n💭 Thought: {thought_3}")
        print(f"🔧 Action: send_alert(level='{obs_2_data['alert_level']}', patient_id='{patient_id}')")

        obs_3 = execute_tool("send_alert", {
            "level": obs_2_data["alert_level"],
            "message": alert_msg,
            "patient_id": patient_id
        })
        obs_3_data = json.loads(obs_3)
        alert_result = obs_3_data
        print(f"👁️  Observation: Alert {obs_3_data['alert_id']} sent to {obs_3_data['recipient']}")

        trace.append({"thought": thought_3, "action": "send_alert", "observation": obs_3_data})

    # ── Final Answer ──
    final_answer = (
        f"CRITICAL HYPOGLYCEMIA DETECTED. "
        f"Patient {patient_id} glucose: {glucose} mg/dL ({trend} trend). "
        f"Alert sent to caregiver. "
        f"Recommended action: {obs_2_data['recommended_action']}"
    )
    print(f"\n✅ Final Answer: {final_answer[:120]}...")

    return {
        "trace": trace,
        "final_answer": final_answer,
        "alert_level": obs_2_data["alert_level"],
        "caregiver_notified": alert_result is not None,
        "steps": len(trace)
    }


print("✅ ReAct agent defined")

In [ ]:
# ─── Live ReAct Agent (requires OpenAI API key) ───────────────────────────────

def run_react_agent_live(patient_id: str, scenario: str) -> dict:
    """Live ReAct agent using llm_utils (Ollama/OpenAI) tool calling."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Monitor patient {patient_id}. Scenario: {scenario}"}
    ]

    sep = "=" * 60
    print(f"\n{sep}")
    print("🤖 GAS Monitor Agent — Live ReAct Trace")
    print(f"   Scenario: {scenario}")
    print(sep)

    trace = []
    max_steps = 5

    for step in range(max_steps):
        response = llm.chat_tools(messages, tools=GAS_TOOLS, tool_choice="auto")
        msg = response.choices[0].message

        if not msg.tool_calls:
            print(f"\n✅ Final Answer: {msg.content}")
            trace.append({"step": step, "type": "final_answer", "content": msg.content})
            return {"trace": trace, "final_answer": msg.content, "steps": step + 1}

        messages.append(msg.model_dump())
        tool_results = []

        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            print(f"\n🔧 Step {step+1}: {fn_name}({fn_args})")
            result = execute_tool(fn_name, fn_args)
            result_data = json.loads(result)
            print(f"👁️  Result: {str(result_data)[:100]}...")
            trace.append({"step": step, "tool": fn_name, "args": fn_args, "result": result_data})
            tool_results.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

        messages.extend(tool_results)

    return {"trace": trace, "final_answer": "Max steps reached", "steps": max_steps}


print("✅ Live ReAct agent defined")

### Step 8: Run the Agent — Critical Hypoglycemia Scenario

In [ ]:
# ─── Run the GAS Monitor Agent ────────────────────────────────────────────────

SCENARIO = "Patient glucose just dropped to 45 mg/dL with a falling trend. This is a medical emergency."

if TEST_MODE:
    print("🧪 Running in TEST_MODE (mock responses)")
    result = run_react_agent_mock("patient-001", SCENARIO)
else:
    print("🔴 Running LIVE with OpenAI API")
    result = run_react_agent_live("patient-001", SCENARIO)

print(f"\n{'='*60}")
print(f"📊 Agent Run Summary")
print(f"{'='*60}")
print(f"  Steps taken: {result['steps']}")
print(f"  Alert level: {result.get('alert_level', 'N/A')}")
print(f"  Caregiver notified: {result.get('caregiver_notified', 'N/A')}")
print(f"  Final answer (truncated): {result['final_answer'][:150]}...")

### Step 9: Initialize GASState

In [ ]:
# ─── Initialize the full GASState ─────────────────────────────────────────────

initial_state: GASState = {
    "patient": alice,
    "current_reading": critical_reading,
    "memory": GASMemory(
        recent_readings=[critical_reading],
        past_episodes=[],
        patient_notes=["Alice has hypoglycemia unawareness — does not feel lows below 60 mg/dL"]
    ),
    "alert_level": result.get("alert_level", "critical"),
    "recommendation": result["final_answer"][:200],
    "safety_approved": False,   # Safety Critic hasn't run yet (Chapter 18)
    "caregiver_notified": result.get("caregiver_notified", True),
    "agent_messages": result["trace"],
    "next_agent": "advisor"     # After Monitor, Supervisor routes to Advisor
}

print("✅ GASState initialized:")
print(f"  patient: {initial_state['patient'].name}, {initial_state['patient'].age}yo {initial_state['patient'].diabetes_type}")
print(f"  current_reading: {initial_state['current_reading'].value} mg/dL ({initial_state['current_reading'].trend})")
print(f"  alert_level: {initial_state['alert_level']}")
print(f"  safety_approved: {initial_state['safety_approved']}")
print(f"  caregiver_notified: {initial_state['caregiver_notified']}")
print(f"  next_agent: {initial_state['next_agent']}")
print(f"  agent_messages: {len(initial_state['agent_messages'])} trace entries")
print(f"\n📝 Recommendation preview:")
print(f"  {initial_state['recommendation'][:200]}")

### Step 10: Visualize the Agent Architecture

In [ ]:
# ─── Print GAS Architecture Summary ──────────────────────────────────────────

GAS_AGENTS = [
    {
        "name": "Supervisor Agent",
        "chapter": "Ch 20",
        "role": "LangGraph orchestrator — routes between agents based on GASState",
        "tools": ["route_to_agent", "check_state", "terminate"],
        "status": "🔜 Coming in Ch 20"
    },
    {
        "name": "Monitor Agent",
        "chapter": "Ch 15 (this chapter)",
        "role": "Reads CGM sensor data, detects anomalies, sets alert_level",
        "tools": ["read_glucose", "check_guidelines", "send_alert"],
        "status": "✅ Built in this chapter"
    },
    {
        "name": "Advisor Agent",
        "chapter": "Ch 16-17",
        "role": "RAG + memory-grounded clinical recommendations",
        "tools": ["retrieve_guidelines", "recall_episodes", "generate_recommendation"],
        "status": "🔜 Coming in Ch 16-17"
    },
    {
        "name": "Safety Critic",
        "chapter": "Ch 18",
        "role": "Reward-model-based output validation — approves/rejects recommendations",
        "tools": ["validate_recommendation", "check_drug_interactions", "flag_unsafe"],
        "status": "🔜 Coming in Ch 18"
    },
    {
        "name": "Caregiver Notifier",
        "chapter": "Ch 25",
        "role": "Escalation to humans via A2A protocol",
        "tools": ["send_sms", "send_push", "call_emergency"],
        "status": "🔜 Coming in Ch 25"
    }
]

print("🏥 Guardian Angel System (GAS) — Agent Roster")
print("=" * 60)
for i, agent in enumerate(GAS_AGENTS, 1):
    print(f"\n{i}. {agent['name']} ({agent['chapter']})")
    print(f"   Role: {agent['role']}")
    print(f"   Tools: {', '.join(agent['tools'])}")
    print(f"   Status: {agent['status']}")

## 🏥 GAS Connection

What we just built is the **skeleton of the GAS Monitor Agent**.

The ReAct loop you see here — perceive glucose → reason about severity → act by alerting — is the core pattern that runs every 5 minutes in the production GAS system.

By the end of Part 5, this simple 3-tool ReAct loop will evolve into:

| Chapter | What Gets Added |
|---------|----------------|
| **Ch 16** | RAG knowledge base — Advisor retrieves ADA guidelines before recommending |
| **Ch 17** | Memory — Advisor remembers Alice's past hypoglycemia episodes |
| **Ch 18** | Harness — GASHarness orchestrates all agents with guardrails |
| **Ch 19** | Loop engineering — Alert loop with stagnation detection and adaptive budget |
| **Ch 20** | LangGraph — Full 5-agent graph with conditional routing |
| **Ch 25** | A2A — Caregiver Notifier uses Agent-to-Agent protocol |

The `GASState` TypedDict you defined in this chapter is the **shared data contract** that makes all of this possible. Every agent reads from and writes to `GASState`.

```python
# The full GAS flow (preview):
state = initial_state
state = monitor_agent(state)      # Ch 15: detect anomaly
state = advisor_agent(state)      # Ch 16-17: retrieve + recommend
state = safety_critic(state)      # Ch 18: validate recommendation
state = caregiver_notifier(state) # Ch 25: escalate if needed
```

## ✅ Checkpoint

### What We Built

| Component | Description |
|-----------|-------------|
| `PatientProfile` | Dataclass for static patient data (Alice, 34, T1D) |
| `GlucoseReading` | Dataclass for CGM readings with trend |
| `GASMemory` | Multi-type memory container |
| `GASState` | Shared TypedDict flowing through all 5 agents |
| `GAS_TOOLS` | 3 OpenAI function-calling tool definitions |
| `run_react_agent_mock` | Full ReAct trace without API key |
| `run_react_agent_live` | Live ReAct agent using GPT-4o-mini |

### Key Concepts

- **Perceive-Reason-Act loop**: The fundamental pattern of all agentic systems
- **ReAct**: Interleaving Thought and Action for transparent reasoning
- **Function calling**: Structured tool invocation via JSON schema
- **GASState**: Shared state contract for multi-agent coordination

### What's Next

**Chapter 16: RAG** — We'll build the GAS Advisor Agent's knowledge layer:
- 20-document medical knowledge base
- Dense + sparse hybrid retrieval
- Reciprocal Rank Fusion (RRF)
- Agentic RAG: the agent decides *when* and *what* to retrieve

---
*Guardian Angel System | Part 5: Agentic AI | Chapter 15*